main changes:
- updated some variable names for readability across functions
- fixed comparison typo in the PLE function to skip the satisfied clauses
- added early conflict detection in the UP logic, and loop restart when a unit clause is found
- some variable caching, handling empty symbols and conflicts, and nesting removal in DPLL driver function

In [ ]:
import json
from pathlib import Path
from argparse import ArgumentParser
from dimacs_parser import DimacsParser
from model_timer import Timer

# import numpy as np

# input_file = '../input/C459_4675.cnf' 
# input_file = '../input/C1597_081.cnf'
input_file = '../input/U50_1065_038.cnf'
path = Path(input_file)
filename = path.name
instance = DimacsParser.parse_cnf_file(input_file)
print(instance, end="")

Number of variables: 50
Number of clauses: 1065
Variables: {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50}
Clause 0: {-18, -14, 30, -36, -34}
Clause 1: {34, -18, -47, -10, -37}
Clause 2: {-24, 9, 11, -15, 30}
Clause 3: {-29, -19, 16, 18, 24}
Clause 4: {-21, 50, 22, -41, -6}
Clause 5: {-27, -13, 20, -2, -1}
Clause 6: {-31, 2, 21, 26, -34}
Clause 7: {-6, -15, -46, 21, 26}
Clause 8: {37, 40, 14, -18, -42}
Clause 9: {-32, 11, -50, -10, 29}
Clause 10: {-30, -23, 11, 12, 27}
Clause 11: {7, -15, -14, -8, -5}
Clause 12: {34, 12, 13, -8, -39}
Clause 13: {-32, -49, -47, 18, 29}
Clause 14: {9, -16, 19, -45, -33}
Clause 15: {-32, -30, 39, 14, 47}
Clause 16: {33, 5, -48, 19, -4}
Clause 17: {-24, 45, 14, -49, -47}
Clause 18: {-30, 40, 43, 46, 14}
Clause 19: {37, -24, 11, 22, -2}
Clause 20: {34, 16, 50, 21, -1}
Clause 21: {-29, 10, 21, -37, 31}
Clause 22: {-3

In [146]:
symbols = list(instance.vars)
clauses = [list(clause) for clause in instance.clauses]
model = {}

In [147]:
def eval_clause(clause, model):
    unassigned = False 

    for var in clause:
        if (abs(var) in model):
            value = model[abs(var)]

            if (var > 0 and value) or (var < 0 and not value):
                return 'TRUE' 
        else: 
            unassigned = True 
    if unassigned:
        return 'UNKNOWN' 
    
    return 'FALSE'

In [148]:
def eval_instance(clauses, model): 
    every = True 
    for clause in clauses: 
        clause_value = eval_clause(clause, model) 

        if clause_value == 'FALSE': 
            return 'UNSAT' 
        if clause_value != 'TRUE': 
            every = False 
    if every:
        return 'SAT' 
    
    return 'UNKNOWN'

In [149]:
def pure_symbol(clauses, model):
    pure = {} 
    impure = set()
    for clause in clauses: 
        if eval_clause(clause, model) == 'TRUE': 
            continue 

        for x in clause: 
            var = abs(x)

            if var in model or var in impure:
                continue

            if var not in pure:
                pure[var] = x > 0

            else: 
                if pure[var] != (x > 0): 
                    pure.pop(var)
                    impure.add(var)
    if not pure: 
        return [], []
    return list(pure.keys()), list(pure.values())

In [150]:
# unit clause 
def unit_clause(clauses, model): 
    model = model.copy() 
    new_assignments ={}
    found_unit_clause = True 

    while found_unit_clause: 
        found_unit_clause = False 

        for clause in clauses:
            clause_val = eval_clause(clause, model)
            if clause_val == 'TRUE':
                continue 
            elif clause_val == 'FALSE':
                return None, None
            
            unassigned = [lit for lit in clause if abs(lit) not in model]

            if len(unassigned) == 1: 
                literal = unassigned[0]
                if (abs(literal)) in model: 
                    continue 
                else: 
                    model[abs(literal)] = literal > 0
                    new_assignments[abs(literal)] = literal > 0
                    found_unit_clause = True

    return list(new_assignments.keys()), list(new_assignments.values())

Pick most frequently occuring variables in unsatisfied clauses

In [151]:
def pick_best_branching_var(symbols, clauses, model):
    var_score = {var: 0 for var in symbols}

    for clause in clauses:
        if eval_clause(clause, model) != 'TRUE':
            for lit in clause:
                var = abs(lit)
                if var in symbols:
                    var_score[var] += 1

    if not var_score:
        return None
    return max(var_score, key=var_score.get)

In [ ]:
def dpll(clauses, symbols, model): 
    
    instance_status = eval_instance(clauses, model)
    if instance_status =='SAT':
        return model 
    elif instance_status =='UNSAT':
        return None 
    
    # unit propagation
    vars, vals = unit_clause(clauses, model)
    if vars is None:
        return None # there's a conflict
    if vars:
        symbols = list(set(symbols) - set(vars))
        model = model | dict(zip(vars, vals))
        return dpll(clauses, symbols, model)
    
    # pure literal elimination
    # vars, vals = pure_symbol(clauses, model) 
    # if vars: 
    #     symbols = list(set(symbols) - set(vars))
    #     model = model | dict(zip(vars, vals))
    #     return dpll(clauses, symbols, model)
    
    # branch
    if not symbols:
        return None

    p = pick_best_branching_var(symbols, clauses, model)
    if p is None:
        return None

    rest = [s for s in symbols if s != p]

    res = dpll(clauses, rest, (model | {p: True}))
    if res is not None:
        return res
    
    return dpll(clauses, rest, (model | {p: False})) # backtracking with False

In [ ]:
result = dpll(clauses, symbols, model)

if result is not None:
    print(f'Result: SAT, Solution: {result}')
else:
    print('UNSAT')

UNSAT
